# FinBERT Multimodal Training on Google Colab

This notebook trains the repo's `finbert_multimodal_attention` model on the weekly S&P 500 dataset.

Use this for the current project setup:

- Dataset name: `sp500_500_2yr`
- Text bundle: SEC filings + Finnhub news + Yahoo RSS news when available
- Target label: `label_abs_alpha_gt_1pct`
- Trading interpretation: positive-only / long-only
- Safe FinBERT max length: `512` tokens

Before running:

1. In Colab, choose `Runtime -> Change runtime type -> T4 GPU`.
2. Upload your repo folder or repo zip to Google Drive.
3. Upload your dataset folder or dataset zip to Google Drive.
4. Edit the path variables in the setup cell below.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## 1. Configure paths and training settings

Recommended Drive layout:

- Repo folder: `/content/drive/MyDrive/2450FinalProject`
- Dataset folder: `/content/drive/MyDrive/2450FinalProject_data/sp500_500_2yr`

Zip files also work. If using zips, set `REPO_SOURCE` or `DATASET_SOURCE` to the `.zip` path.

The dataset source should contain files like:

- `weekly_event_dataset.csv` or `.parquet`
- `price_features.csv` or `.parquet`
- `sec_filings_text.csv` or `.parquet`
- `finnhub_news.csv` or `.parquet`
- `yahoo_news.csv` or `.parquet`
- `coverage_overall.json`

In [ ]:
from pathlib import Path

# Change these if your Drive paths are different.
REPO_SOURCE = Path('/content/drive/MyDrive/2450FinalProject')
DATASET_SOURCE = Path('/content/drive/MyDrive/2450FinalProject_data/sp500_500_2yr')

# Fast local working copy inside the Colab VM.
WORKDIR = Path('/content/2450FinalProject')
DATASET_NAME = 'sp500_500_2yr'

# Persistent caches/backups in Drive.
HF_CACHE_DIR = Path('/content/drive/MyDrive/hf_cache')
ARTIFACT_BACKUP_DIR = Path('/content/drive/MyDrive/2450FinalProject_colab_artifacts')

# Training configuration. Keep MAX_LENGTH at 512 because FinBERT/BERT cannot use 1024 directly.
EPOCHS = 10
BATCH_SIZE = 8
MAX_LENGTH = 512
LEARNING_RATE = 2e-4
CHECKPOINT_METRIC = 'average_precision'
THRESHOLD_METRIC = 'f1'
ALPHA_TRADE_OBJECTIVE = 'return_on_traded_capital'
INCLUDE_TICKER = True
TRAIN_ON_TEXT_ONLY = False
LOCAL_FILES_ONLY = False
UNFREEZE_FINBERT = False

# If you OOM on T4, set BATCH_SIZE = 4 or 2. Do not set MAX_LENGTH > 512 for this FinBERT checkpoint.
assert MAX_LENGTH <= 512, 'ProsusAI/finbert is BERT-based and only supports up to 512 tokens.'

HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_BACKUP_DIR.mkdir(parents=True, exist_ok=True)
print('Repo source:', REPO_SOURCE)
print('Dataset source:', DATASET_SOURCE)
print('Workdir:', WORKDIR)

## 2. Copy repo and dataset into `/content`

Training directly from Drive is much slower. This cell copies the repo and dataset to the Colab VM.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path


def find_repo_root(root: Path) -> Path:
    candidates = [root, *root.rglob('*')]
    for candidate in candidates:
        if candidate.is_dir() and (candidate / 'scripts' / 'train_multimodal_attention.py').exists():
            return candidate
    raise FileNotFoundError(f'Could not find repo root under {root}')


def copy_tree_clean(src: Path, dst: Path, ignore_names: tuple[str, ...] = ()) -> None:
    if dst.exists():
        shutil.rmtree(dst)
    ignore = shutil.ignore_patterns(*ignore_names) if ignore_names else None
    shutil.copytree(src, dst, ignore=ignore)


def extract_zip(src_zip: Path, dst_root: Path) -> Path:
    if dst_root.exists():
        shutil.rmtree(dst_root)
    dst_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src_zip, 'r') as zf:
        zf.extractall(dst_root)
    return dst_root


# Copy or extract repo.
if not REPO_SOURCE.exists():
    raise FileNotFoundError(f'REPO_SOURCE does not exist: {REPO_SOURCE}')

if REPO_SOURCE.is_file() and REPO_SOURCE.suffix.lower() == '.zip':
    extracted = extract_zip(REPO_SOURCE, Path('/content/_repo_extract'))
    repo_root = find_repo_root(extracted)
else:
    repo_root = find_repo_root(REPO_SOURCE)

copy_tree_clean(
    repo_root,
    WORKDIR,
    ignore_names=('2450venv', '.git', '__pycache__', '.pytest_cache', '.next', 'node_modules'),
)
print('Copied repo to:', WORKDIR)

# Copy or extract dataset into WORKDIR/data/DATASET_NAME.
target_dataset_dir = WORKDIR / 'data' / DATASET_NAME
target_dataset_dir.parent.mkdir(parents=True, exist_ok=True)

if DATASET_SOURCE.exists():
    if DATASET_SOURCE.is_file() and DATASET_SOURCE.suffix.lower() == '.zip':
        extracted_data = extract_zip(DATASET_SOURCE, Path('/content/_dataset_extract'))
        possible_dataset_dirs = [p for p in [extracted_data, *extracted_data.rglob('*')] if p.is_dir() and ((p / 'weekly_event_dataset.csv').exists() or (p / 'weekly_event_dataset.parquet').exists())]
        if not possible_dataset_dirs:
            raise FileNotFoundError('Could not find weekly_event_dataset inside dataset zip.')
        dataset_root = possible_dataset_dirs[0]
    else:
        dataset_root = DATASET_SOURCE

    copy_tree_clean(dataset_root, target_dataset_dir)
    print('Copied dataset to:', target_dataset_dir)
else:
    print('DATASET_SOURCE not found. Checking whether dataset already exists in repo copy...')
    if not target_dataset_dir.exists():
        raise FileNotFoundError(f'No dataset found at {DATASET_SOURCE} or {target_dataset_dir}')

os.chdir(WORKDIR)
print('Current working directory:', Path.cwd())

## 3. Install dependencies

This uses the repo's `requirements.txt`. If Colab already has compatible packages, this is still fine.

After installing, Colab may ask you to restart the runtime if core packages changed. If that happens, restart and rerun cells from the top.

In [ ]:
%pip install -q --upgrade pip
%pip install -q -r requirements.txt
%pip install -q sentencepiece accelerate

## 4. GPU and dataset sanity checks

This verifies CUDA, checks the hard FinBERT length limit, and prints dataset coverage.

In [ ]:
import json
import os
from pathlib import Path

import pandas as pd
import torch

os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE_DIR)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU memory GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError('No CUDA GPU detected. In Colab, switch Runtime -> Change runtime type -> T4 GPU.')

coverage_path = Path('data') / DATASET_NAME / 'coverage_overall.json'
if coverage_path.exists():
    coverage = json.loads(coverage_path.read_text())
    print('\nCoverage summary:')
    print(json.dumps(coverage, indent=2))
else:
    print('No coverage_overall.json found; continuing.')

required_any = [
    'weekly_event_dataset',
    'price_features',
    'sec_filings_text',
    'finnhub_news',
    'yahoo_news',
]
for stem in required_any:
    csv_exists = (Path('data') / DATASET_NAME / f'{stem}.csv').exists()
    parquet_exists = (Path('data') / DATASET_NAME / f'{stem}.parquet').exists()
    print(f'{stem}:', 'ok' if csv_exists or parquet_exists else 'missing')

## 5. Prepare chronological split

This rebuilds `artifacts/splits/sp500_500_2yr` from the weekly dataset and records class balance metadata.

In [ ]:
import json
import subprocess
from pathlib import Path

split_dir = Path('artifacts') / 'splits' / DATASET_NAME
input_dir = Path('data') / DATASET_NAME

prepare_cmd = [
    'python3',
    'scripts/prepare_train_test_split.py',
    '--input-dir', str(input_dir),
    '--dataset-name', DATASET_NAME,
]
print('Running:', ' '.join(prepare_cmd))
subprocess.run(prepare_cmd, check=True)

split_metadata = json.loads((split_dir / 'split_metadata.json').read_text())
print('\nSplit metadata:')
for key in [
    'total_rows', 'train_rows', 'test_rows',
    'train_start', 'train_end', 'test_start', 'test_end',
    'label_column', 'full_positive_rate', 'train_positive_rate', 'test_positive_rate',
    'full_class_counts', 'train_class_counts', 'test_class_counts',
]:
    print(f'{key}: {split_metadata.get(key)}')

## 6. Train FinBERT multimodal attention

Default settings are T4-safe:

- `MAX_LENGTH = 512`
- `BATCH_SIZE = 8`
- FinBERT frozen by default

If you run out of memory, lower `BATCH_SIZE` to `4` or `2`.

If you want actual end-to-end FinBERT fine-tuning, set `UNFREEZE_FINBERT = True` and use `BATCH_SIZE = 2` or `4`.

In [ ]:
import subprocess
from pathlib import Path

output_dir = Path('artifacts') / 'experiments' / 'finbert_multimodal_attention' / f'{DATASET_NAME}_colab'
split_dir = Path('artifacts') / 'splits' / DATASET_NAME
input_dir = Path('data') / DATASET_NAME

train_cmd = [
    'python3',
    'scripts/train_multimodal_attention.py',
    '--split-dir', str(split_dir),
    '--dataset-name', DATASET_NAME,
    '--input-dir', str(input_dir),
    '--output-dir', str(output_dir),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--max-length', str(MAX_LENGTH),
    '--learning-rate', str(LEARNING_RATE),
    '--checkpoint-metric', CHECKPOINT_METRIC,
    '--threshold-metric', THRESHOLD_METRIC,
    '--alpha-trade-objective', ALPHA_TRADE_OBJECTIVE,
]

if INCLUDE_TICKER:
    train_cmd.append('--include-ticker')
if TRAIN_ON_TEXT_ONLY:
    train_cmd.append('--train-on-text-only')
if LOCAL_FILES_ONLY:
    train_cmd.append('--local-files-only')
if UNFREEZE_FINBERT:
    train_cmd.append('--unfreeze-finbert')

print('Running:')
print(' '.join(train_cmd))
subprocess.run(train_cmd, check=True)

## 7. Inspect saved metrics and long-only trading summary

In [ ]:
import json
from pathlib import Path

run_dir = Path('artifacts') / 'experiments' / 'finbert_multimodal_attention' / f'{DATASET_NAME}_colab'
metrics = json.loads((run_dir / 'metrics.json').read_text())
metadata = json.loads((run_dir / 'metadata.json').read_text())
trading_summary = json.loads((run_dir / 'trading_summary.json').read_text())

print('Metrics:')
print(json.dumps(metrics, indent=2))

print('\nKey metadata:')
for key in [
    'model_name', 'dataset_name', 'label_column',
    'trading_mode', 'signal_regime', 'portfolio_type',
    'train_rows', 'val_rows', 'test_rows',
    'full_positive_rate', 'train_positive_rate', 'test_positive_rate',
    'text_coverage_train', 'text_coverage_test',
    'threshold', 'alpha_trade_threshold',
    'best_checkpoint_epoch',
]:
    print(f'{key}: {metadata.get(key)}')

print('\nTrading summary:')
print(json.dumps(trading_summary, indent=2))

## 8. Back up artifacts to Drive

This copies the full `artifacts/` directory to Drive and also creates a zip archive.

In [ ]:
import shutil
from pathlib import Path

src_artifacts = Path('artifacts')
dst_artifacts = ARTIFACT_BACKUP_DIR / f'{DATASET_NAME}_finbert_colab_artifacts'

if dst_artifacts.exists():
    shutil.rmtree(dst_artifacts)
shutil.copytree(src_artifacts, dst_artifacts)

zip_base = ARTIFACT_BACKUP_DIR / f'{DATASET_NAME}_finbert_colab_artifacts'
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=dst_artifacts)

print('Artifacts folder copied to:', dst_artifacts)
print('Artifacts zip written to:', zip_path)

## Troubleshooting

### `max-length 1024` crash

`ProsusAI/finbert` is BERT-based and only supports 512 tokens. The tokenizer truncates text; it does not compress it. Use `MAX_LENGTH = 512`.

### CUDA out of memory

Use one of these:

- `BATCH_SIZE = 4`, `MAX_LENGTH = 512`
- `BATCH_SIZE = 2`, `MAX_LENGTH = 512`
- Keep `UNFREEZE_FINBERT = False`

### First run cannot find FinBERT weights

Keep `LOCAL_FILES_ONLY = False` on the first run. After the model is cached in `HF_CACHE_DIR`, repeat runs can use `LOCAL_FILES_ONLY = True`.

### Yahoo RSS rows are zero

That is from Yahoo 429 rate limiting. It is not fatal. SEC + Finnhub still train. If you regenerate data and want Yahoo, rerun the downloader with fewer news workers, e.g. `--news-workers 1`.

### Dataset has fewer than 52,000 rows

That is expected after price availability, weekly merge, missing labels, and chronology filtering. Use the actual `weekly_event_dataset` and split metadata in your report.